In [1]:
# import libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
import json

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
# mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Define Drive and local dataset paths

PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_CNN_Project"

# Original dataset remains safely stored in Google Drive
DRIVE_DATASET_DIR = os.path.join(
    PROJECT_DIR,
    "dataset",
    "Epic and CSCR hospital Dataset"
)

# Model and result files should remain on Drive so they survive runtime resets
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Drive dataset exists:", os.path.isdir(DRIVE_DATASET_DIR))
assert os.path.isdir(DRIVE_DATASET_DIR), (
    f"Dataset folder not found in Google Drive: {DRIVE_DATASET_DIR}"
)


Project directory: /content/drive/MyDrive/Brain_Tumor_CNN_Project
Drive dataset exists: True


In [4]:
# Constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

# Class Names for Binary Classification
CLASS_NAMES = ['Non-Tumourous', 'Tumourous']

print("Configuration Loaded.")

Configuration Loaded.


In [5]:
import shutil
import os

# 1. Define the source path (Google Drive) and destination path (Local Colab Storage)
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Brain_Tumor_CNN_Project/dataset/Epic and CSCR hospital Dataset"
LOCAL_DATASET_DIR = "/content/brain_tumor_dataset"

# 2. Copy the dataset if it doesn't already exist locally
if not os.path.exists(LOCAL_DATASET_DIR):
    print("Copying dataset to Colab local storage for faster access...")
    try:
        # copytree copies the entire directory structure
        shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
        print("Dataset copied successfully!")
    except Exception as e:
        print(f"Error copying dataset: {e}")
else:
    print("Local dataset already exists. Skipping copy.")

# 3. Update your directory paths to point to the LOCAL storage
# These are the paths you will use for loading images in your preprocessing function
DATASET_DIR = LOCAL_DATASET_DIR
TRAIN_DIR = os.path.join(DATASET_DIR, "Train")
TEST_DIR = os.path.join(DATASET_DIR, "Test")

# Verify paths exist
print(f"Local dataset path: {DATASET_DIR}")
print(f"Train directory exists: {os.path.exists(TRAIN_DIR)}")
print(f"Test directory exists: {os.path.exists(TEST_DIR)}")

Copying dataset to Colab local storage for faster access...
Dataset copied successfully!
Local dataset path: /content/brain_tumor_dataset
Train directory exists: True
Test directory exists: True


In [7]:
# Load the CSVs created in 02_Preprocessing.ipynb
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "validation_split.csv")) # Note: Check if your file is named validation_split or val_split
test_df = pd.read_csv(os.path.join(RESULTS_DIR, "test_split.csv"))

# Verify counts
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

# CRITICAL FIX: Ensure test set size matches expectation
assert len(test_df) == 2093, f"Expected 2415 test images, found {len(test_df)}"

# Display class distribution
print("\nTrain Distribution:")
print(train_df['label'].value_counts())

Training samples: 4844
Validation samples: 1930
Test samples: 2093

Train Distribution:
label
1    4169
0     675
Name: count, dtype: int64


In [8]:
# Calculate weights to handle imbalance (Non-Tumourous is minority)
y_train = train_df['label'].values
classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = {int(label): float(weight) for label, weight in zip(classes, class_weights_array)}

print("Class Weights Calculated:")
for label, weight in class_weights.items():
    print(f"{CLASS_NAMES[label]} (Class {label}): {weight:.4f}")

Class Weights Calculated:
Non-Tumourous (Class 0): 3.5881
Tumourous (Class 1): 0.5810


In [9]:
def preprocess_image(image_path, label):
    """Loads and normalizes an image."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0  # Normalize to [0, 1]
    return img, label

def augment(image, label):
    """Applies random augmentation to training images."""
    # Random Flip
    image = tf.image.random_flip_left_right(image)
    # Random Rotation (small angle to keep anatomy realistic)
    image = tf.image.rot90(image, k=tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32))
    # Random Brightness
    image = tf.image.random_brightness(image, max_delta=0.1)
    # Random Contrast
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image, label

print("Preprocessing functions defined.")

Preprocessing functions defined.


In [10]:
# --- TRAINING DATASET (Shuffled + Augmented) ---
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df['image_path'].values, train_df['label'].values)
)
train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE) # Apply Augmentation
train_dataset = train_dataset.shuffle(buffer_size=1000, seed=SEED)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [11]:
# --- VALIDATION DATASET (No Shuffle, No Augment) ---
val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_df['image_path'].values, val_df['label'].values)
)
val_dataset = val_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

In [12]:
# --- TEST DATASET (NO SHUFFLE - Critical for Evaluation) ---
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df['image_path'].values, test_df['label'].values)
)
test_dataset = test_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
# DO NOT SHUFFLE TEST DATA
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

print("Data pipelines created successfully.")

Data pipelines created successfully.


In [13]:
model = models.Sequential([
    # Input Layer
    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 4
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Classifier Head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid') # Binary Output
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

#### CNN Architecture Components

1. **Conv2D layers** learn visual features from the MRI images.

2. The number of filters increases from **32 → 64 → 128 → 256**, allowing the network to learn increasingly complex features.

3. **MaxPooling2D** reduces spatial dimensions while retaining important features.

4. **BatchNormalization** helps stabilize and speed up training.

5. **GlobalAveragePooling2D** reduces the number of parameters compared with a large `Flatten()` layer.

6. **Dropout(0.5)** helps reduce overfitting.

7. The final **Sigmoid** neuron produces a value between 0 and 1 for our binary classification:

   * closer to `0` → **Non-Tumourous**
   * closer to `1` → **Tumourous**


In [14]:
# Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

In [15]:
# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "best_cnn_model.h5"),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# Train
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7583 - loss: 0.5916 - precision: 0.9371 - recall: 0.7716
Epoch 1: val_accuracy improved from None to 0.79845, saving model to /content/drive/MyDrive/Brain_Tumor_CNN_Project/models/best_cnn_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Brain_Tumor_CNN_Project/models/best_cnn_model.h5
152/152 ━━━━━━━━━━━━━━━━━━━━ 1372s 9s/step - accuracy: 0.7993 - loss: 0.5212 - precision: 0.9491 - recall: 0.8103 - val_accuracy: 0.7984 - val_loss: 0.6537 - val_precision: 0.7984 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.8496 - loss: 0.4403 - precision: 0.9567 - recall: 0.8635
Epoch 2: val_accuracy did not improve from 0.79845
152/152 ━━━━━━━━━━━━━━━━━━━━ 1288s 8s/step - accuracy: 0.8623 - loss: 0.4017 - precision: 0.9637 - recall: 0.8729 - val_accuracy: 0.7984 - val_loss: 1.0851 - val_precision: 0.7984 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/20
152/152 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.8642 - loss: 0.4171 - precision: 0.9638 - recall: 0.8755
Epoch 3: val_accuracy did not improve from 0.79845
152/152 ━━━━━━━━━━━━━━━━━━━━ 1274s 8s/step - accuracy: 0.8747 - loss: 0.3653 - precision: 0.9717 


Epoch 4: finished saving model to /content/drive/MyDrive/Brain_Tumor_CNN_Project/models/best_cnn_model.h5
152/152 ━━━━━━━━━━━━━━━━━━━━ 1288s 8s/step - accuracy: 0.8788 - loss: 0.3553 - precision: 0.9726 - recall: 0.8841 - val_accuracy: 0.8544 - val_loss: 0.3419 - val_precision: 0.9375 - val_recall: 0.8761 - learning_rate: 0.0010
Epoch 5/20
 23/152 ━━━━━━━━━━━━━━━━━━━━ 16:04 7s/step - accuracy: 0.8225 - loss: 0.3784 - precision: 0.9771 - recall: 0.8189

In [ ]:
# Evaluate the best CNN model on the test dataset

test_results = model.evaluate(test_dataset, verbose=1)

print("\nTest evaluation completed.")

In [ ]:
# Generate predictions for the complete test dataset

y_prob = model.predict(test_dataset, verbose=1)

print("Predictions generated successfully.")
print("Prediction shape:", y_prob.shape)

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')

    plt.tight_layout()
    plt.show()

plot_history(history)

- **Adam optimizer** — adjusts the model's weights efficiently during training.
- **Learning rate** = 0.001 — provides a reasonable starting learning rate for our CNN.
- **Binary crossentropy** — appropriate because our task has two classes: Tumourous and Non-Tumourous.
- **Accuracy** — measures the overall proportion of correct predictions.
- **Precision** — measures how many images predicted as tumourous are actually tumourous.
- **Recall** — measures how many actual tumourous images are correctly detected.

In [ ]:
# Evaluate on Test Set
print("Evaluating on Test Set...")
test_results = model.evaluate(test_dataset, verbose=1)

# Unpack results based on metrics: [loss, accuracy, precision, recall]
test_loss = test_results[0]
test_acc = test_results[1]
test_prec = test_results[2]
test_rec = test_results[3]

print(f"\n--- Final Test Metrics ---")
print(f"Loss:     {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision:{test_prec:.4f}")
print(f"Recall:   {test_rec:.4f}")

# Save metrics
metrics_df = pd.DataFrame([{
    'Accuracy': test_acc,
    'Precision': test_prec,
    'Recall': test_rec,
    'F1 Score': 2 * (test_prec * test_rec) / (test_prec + test_rec) if (test_prec + test_rec) > 0 else 0
}])
metrics_df.to_csv(os.path.join(RESULTS_DIR, "test_metrics.csv"), index=False)

In [ ]:
# Get Predictions (Ensure order matches test_df because we didn't shuffle)
y_prob = model.predict(test_dataset)
y_pred = (y_prob >= 0.5).astype(int).flatten()
y_true = test_df['label'].values

# Classification Report
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

# Save Report
with open(os.path.join(RESULTS_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Save Predictions for Streamlit App
prediction_results = pd.DataFrame({
    'image_path': test_df['image_path'].values,
    'true_label': y_true,
    'predicted_probability': y_prob.flatten(),
    'predicted_label': y_pred
})
prediction_results.to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)
print("All evaluation artifacts saved.")